In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow import keras
from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, Lambda, BatchNormalization, Dropout


(x_train, y_train), (x_test, y_test) = mnist.load_data()

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test  = np.reshape(x_test,  (len(x_test),  28, 28, 1))


hidden_dim = 2
batch_size = 60 # должно быть кратно 60 000

def dropout_and_batch(x):
    return Dropout(0.3)(BatchNormalization()(x))

input_img = Input((28, 28, 1))
x = Flatten()(input_img)
x = Dense(256, activation='relu')(x)
x = dropout_and_batch(x)
x = Dense(128, activation='relu')(x)
x = dropout_and_batch(x)

z_mean = Dense(hidden_dim)(x)
z_log_var = Dense(hidden_dim)(x)

def noiser(args):
    global z_mean, z_log_var
    z_mean, z_log_var = args
    N = tf.random.normal(shape=(batch_size, hidden_dim), mean=0., stddev=1.0)
    return tf.exp(z_log_var / 2) * N + z_mean

h = Lambda(noiser, output_shape=(hidden_dim,))([z_mean, z_log_var])

input_dec = Input(shape=(hidden_dim,))
d = Dense(128, activation='relu')(input_dec)
d = dropout_and_batch(d)
d = Dense(256, activation='relu')(d)
d = dropout_and_batch(d)
d = Dense(28*28, activation='sigmoid')(d)
decoded = Reshape((28, 28, 1))(d)

encoder = keras.Model(input_img, h, name='encoder')
decoder = keras.Model(input_dec, decoded, name='decoder')
vae = keras.Model(input_img, decoder(encoder(input_img)), name="vae")


def vae_loss(x, y):
    x = tf.reshape(x, shape=(batch_size, 28*28))
    y = tf.reshape(y, shape=(batch_size, 28*28))
    loss = tf.reduce_sum(tf.square(x-y), axis=-1)
    kl_loss = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
    return loss + kl_loss

vae.compile(optimizer='adam', loss=vae_loss)

vae.fit(x_train, x_train, epochs=5, batch_size=batch_size, shuffle=True)


h = encoder.predict(x_test[:6000], batch_size=batch_size)
plt.scatter(h[:, 0], h[:, 1])


n = 5
total = 2*n+1

plt.figure(figsize=(total/2, total/2))

num = 1
for i in range(-n, n+1):
    for j in range(-n, n+1):
        ax = plt.subplot(total, total, num)
        num += 1
        img = decoder.predict(np.expand_dims([3*i/n, 3*j/n], axis=0))
        plt.imshow(img.squeeze(), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)

## Izah

In [ ]:
import numpy as np
# NumPy — massivlərlə işləmək üçün.
# Bu kodda:
# - np.reshape      → massivin formasını dəyişmək (28x28 → 28x28x1)
# - np.expand_dims  → massivə əlavə ox əlavə etmək: (2,) → (1, 2)


import matplotlib.pyplot as plt
# matplotlib — şəkilləri və qrafikləri göstərmək üçün.
# Bu kodda:
# - plt.scatter  → nöqtə buludu (gizli fəzanın xəritəsi)
# - plt.figure   → yeni qrafik pəncərəsi açmaq
# - plt.subplot  → bir pəncərədə şəkil şəbəkəsi (cədvəl) düzəltmək
# - plt.imshow   → massivi şəkil kimi göstərmək


import tensorflow as tf
# TensorFlow-un özü. Əvvəlki kodlarda yox idi, burada LAZIMDIR, çünki
# xüsusi əməliyyatları (təsadüfi ədəd, exp, kvadrat, cəm) tf ilə yazırıq:
#   tf.random.normal   → normal paylanmadan təsadüfi ədəd (səs-küy)
#   tf.exp, tf.square  → eksponent, kvadrat
#   tf.reduce_sum      → cəmləmək
#   tf.reshape         → forma dəyişmək


from tensorflow.keras.datasets import mnist
# MNIST — əl ilə yazılmış rəqəmlər (0..9).
#   - 60000 öyrətmə şəkli, 10000 test şəkli
#   - hər şəkil 28x28 piksel, boz tonlu, piksel qiyməti 0..255


from tensorflow import keras
# Keras-ın özü. keras.Model(...) üçün lazımdır.


from tensorflow.keras.layers import Dense, Flatten, Reshape, Input, Lambda, BatchNormalization, Dropout
# Dense              → tam bağlı qat
# Flatten            → (28, 28, 1) → (784,)
# Reshape            → (784,) → (28, 28, 1)
# Input              → modelin giriş formasını elan edir
# Lambda             → ƏVVƏLKİ KODLARDA YOX İDİ. Öz yazdığımız funksiyanı
#                      "qat" kimi şəbəkəyə qoşmağa imkan verir (burada: səs-küy əlavə edən qat).
# BatchNormalization → hər batch-də qatın çıxışını normallaşdırır (orta≈0, yayılma≈1).
#                      Öyrənməni sürətləndirir və sabitləşdirir.
# Dropout            → öyrətmə zamanı neyronların bir hissəsini təsadüfi söndürür.
#                      Əzbərləməyə (overfitting) qarşı.


# =====================================================================
# BU KOD NƏDİR? — VARIATIONAL AUTOENCODER (VAE, "variasiyalı avtoenkoder")
# =====================================================================
#
# Əvvəlki 2 ölçülü autoenkoderlə (AE) müqayisədə ƏSAS YENİLİK:
#
#   Adi AE:  şəkil ──► encoder ──► 2 ədəd (bir NÖQTƏ) ──► decoder ──► şəkil
#   VAE:     şəkil ──► encoder ──► (orta, yayılma) ──► NÖQTƏ ətrafında
#                                  təsadüfi seçim ──► decoder ──► şəkil
#
# Yəni VAE hər şəkli bir NÖQTƏ ilə yox, bir "BULUD" (normal paylanma) ilə
# təsvir edir:
#   - z_mean    → buludun mərkəzi (2 ədəd)
#   - z_log_var → buludun enini xarakterizə edir (2 ədəd, loqarifm şəklində)
# Sonra bu buluddan TƏSADÜFİ bir nöqtə seçilir və decoder-ə verilir.
#
# NİYƏ BELƏ ETMƏK LAZIMDIR?
#   Adi AE-də gizli fəzada "boşluqlar" qalır: öyrətmə nöqtələri arasında
#   decoder-in heç vaxt görmədiyi yerlər var. Orada nöqtə seçsək, nəticə
#   mənasız lək olur (əvvəlki koddakı [50, 250] problemi məhz bu idi).
#   VAE isə nöqtələri məcbur edir ki:
#     1) bir-birinə yaxın dursunlar (boşluq qalmasın),
#     2) 0 ətrafında, standart normal paylanma kimi (mərkəz 0, yayılma 1) yığılsınlar.
#   Nəticə: gizli fəzanın [-3, 3] aralığında istənilən nöqtə seçsək,
#   decoder ona uyğun düzgün rəqəm çəkir. Bu, həqiqi GENERASİYADIR.
#
# ƏVVƏLKİ KODLA MÜQAYİSƏ:
#
#                          2 ölçülü AE            Bu kod (VAE)
# Darboğaz:                2 ədəd (nöqtə)         2 ədəd (təsadüfi seçilmiş nöqtə)
# Encoder çıxışı:          sabit                  hər çağırışda bir az fərqli (səs-küy var)
# Loss:                    yalnız MSE             bərpa xətası + KL (nizamlayıcı)
# Gizli fəzanın miqyası:   nəzarətsiz ([50,250]…) nəzarətli (təxminən [-3, 3])
# Boşluqlar:               var                    azdır
# BatchNorm / Dropout:     yox                    var
# Epochs / batch:          10 / 64                5 / 60
# Yeni şəkil generasiyası: etibarsız              etibarlı (şəbəkə üzrə baxış)
#
#   şəkil (784) ─► [ENCODER] ─► z_mean, z_log_var ─► (səs-küy) ─► h (2 ədəd)
#                                                                    │
#                                                                    ▼
#                                                             [DECODER] ─► şəkil (784)


# ---------------------------------------------------------------------
# 1) DATANIN YÜKLƏNMƏSİ VƏ HAZIRLANMASI
# ---------------------------------------------------------------------

(x_train, y_train), (x_test, y_test) = mnist.load_data()
# Dörd massiv:
#   x_train → (60000, 28, 28), uint8 (0..255)
#   y_train → (60000,) rəqəmlər (bu kodda öyrətmədə İSTİFADƏ OLUNMUR)
#   x_test  → (10000, 28, 28)
#   y_test  → (10000,)  (bu kodda istifadə olunmur)

# стандартизация входных данных
x_train = x_train / 255
x_test = x_test / 255
# (Rusca şərh: "giriş datasının standartlaşdırılması".)
# Piksellər 0..255 → 0..1. Səbəb: şəbəkə kiçik ədədlərlə yaxşı öyrənir və
# decoder-in sigmoid çıxışı da 0..1 aralığındadır, yəni bərpa ilə orijinal
# eyni miqyasda müqayisə olunur.

x_train = np.reshape(x_train, (len(x_train), 28, 28, 1))
x_test  = np.reshape(x_test,  (len(x_test),  28, 28, 1))
# Kanal ölçüsü əlavə edir:
#   (60000, 28, 28) → (60000, 28, 28, 1)
#   (10000, 28, 28) → (10000, 28, 28, 1)
# Son 1 → tək kanal (boz ton). Elementlərin sayı dəyişmir.


# ---------------------------------------------------------------------
# 2) PARAMETRLƏR
# ---------------------------------------------------------------------

hidden_dim = 2
# Gizli fəzanın ölçüsü = 2. Əvvəlki kimi: 2 ədəd müstəvidə nöqtə kimi
# çəkilə bilir (görmək üçün). Keyfiyyət üçün yox, GÖRSƏNMƏ üçündür.

batch_size = 60 # должно быть кратно 60 000
# (Rusca şərh: "60 000-ə bölünməlidir".)
# Hər addımda 60 şəkil. 60000 / 60 = 1000 addım hər epoch-da, qalıq YOXDUR.
#
# NİYƏ BÖLÜNMƏLİDİR? Bu kodda iki yerdə batch ölçüsü SABİT olaraq istifadə olunur:
#   1) noiser-də:   tf.random.normal(shape=(batch_size, hidden_dim))
#   2) vae_loss-da: tf.reshape(x, shape=(batch_size, 28*28))
# Əgər sonuncu batch yarımçıq olsaydı (məsələn 32 şəkil), forma uyğun
# gəlməzdi və xəta çıxardı. Əvvəlki kodlarda (64) bu problem yox idi,
# çünki orada batch ölçüsü sabit yazılmırdı.
# Eyni səbəbdən predict-də də batch_size verilir, 6000 də 60-a bölünür.


# ---------------------------------------------------------------------
# 3) KÖMƏKÇİ FUNKSİYA
# ---------------------------------------------------------------------

def dropout_and_batch(x):
    return Dropout(0.3)(BatchNormalization()(x))
# Yazını qısaltmaq üçün: x-i əvvəl BatchNormalization-dan, sonra
# Dropout-dan keçirir. Sıra: Dense → BatchNorm → Dropout.
#
# BatchNormalization: hər neyronun çıxışını batch üzrə normallaşdırır.
#   Parametr: hər neyron üçün 4 ədəd (2-si öyrənilir, 2-si statistika).
# Dropout(0.3): öyrətmə zamanı neyronların 30%-ni təsadüfi 0 edir.
#   Parametr: yoxdur. Test/predict zamanı AVTOMATİK söndürülür.
#
# Hər çağırışda YENİ qatlar yaradılır, ona görə bu funksiyanı bir neçə dəfə
# çağırmaq təhlükəsizdir (qatlar paylaşılmır).


# ---------------------------------------------------------------------
# 4) ENCODER HİSSƏSİ
# ---------------------------------------------------------------------
#
#   Giriş       Flatten    Dense+BN+Drop   Dense+BN+Drop
#  (28,28,1) ─► (784,) ─► (256,) ─► (128,) ─┬─► z_mean    (2,)
#                                            └─► z_log_var (2,)
#                                                    │
#                                                    ▼
#                                      noiser (Lambda) ─► h (2,)

input_img = Input((28, 28, 1))
# Giriş yeri: forma (28, 28, 1). Batch ölçüsü yazılmır.

x = Flatten()(input_img)
# (28, 28, 1) → (784,). Parametr: yoxdur.
# DİQQƏT: bu x şəbəkənin cari nöqtəsidir, x_train ilə qarışdırmayın.

x = Dense(256, activation='relu')(x)
# 784 → 256, ReLU. Parametr: 784 × 256 + 256 = 200960

x = dropout_and_batch(x)
# 256 ölçüsü saxlanır. BatchNorm: 4 × 256 = 1024 parametr (512-si öyrənilir).

x = Dense(128, activation='relu')(x)
# 256 → 128, ReLU. Parametr: 256 × 128 + 128 = 32896

x = dropout_and_batch(x)
# BatchNorm: 4 × 128 = 512 parametr.

z_mean = Dense(hidden_dim)(x)
# 128 → 2. Aktivasiya YOXDUR (linear). Parametr: 128 × 2 + 2 = 258.
# Mənası: şəklin gizli fəzadakı "bulud"unun MƏRKƏZİ (2 koordinat).
# Aktivasiya qoymuruq, çünki mərkəz mənfi də, müsbət də ola bilməlidir.

z_log_var = Dense(hidden_dim)(x)
# 128 → 2. Aktivasiya YOXDUR. Parametr: 258.
# Mənası: buludun yayılmasının LOQARİFMİ (log σ²), 2 koordinat üçün.
#
# NİYƏ σ² yox, log(σ²)? Dispersiya həmişə müsbət olmalıdır. Şəbəkə isə
# istənilən ədəd çıxara bilir. Loqarifm yazanda isə hər ədəd (mənfi də)
# uyğundur: σ² = exp(log σ²) həmişə müsbətdir. Bu, həm də sabitliyi artırır.


def noiser(args):
    global z_mean, z_log_var
    z_mean, z_log_var = args
    N = tf.random.normal(shape=(batch_size, hidden_dim), mean=0., stddev=1.0)
    return tf.exp(z_log_var / 2) * N + z_mean
# BU, VAE-nin ürəyidir: "REPARAMETRİZASİYA HİLƏSİ".
#
#   args → [z_mean, z_log_var] siyahısı (Lambda qatı belə verir).
#
#   global z_mean, z_log_var:
#     Funksiyanın içində z_mean və z_log_var-ı GLOBAL dəyişənlərə yenidən
#     mənimsədirik. Məqsəd: aşağıdakı vae_loss funksiyası bu dəyişənlərə
#     çata bilsin (KL hissəsi üçün lazımdır). Yəni bu, loss-a məlumat
#     ötürmək üçün bir "qısa yoldur". İşləyir, amma təmiz üsul deyil:
#     funksiya "gizli" yan təsir edir. (Daha təmiz variant: xüsusi Layer
#     yazıb add_loss istifadə etmək.)
#
#   N = tf.random.normal(...):
#     Forma (60, 2): hər şəkil üçün 2 təsadüfi ədəd, orta 0, σ = 1.
#     Bu, standart normal səs-küydür.
#
#   return exp(z_log_var / 2) * N + z_mean:
#     exp(log σ² / 2) = exp(log σ) = σ  (standart sapma).
#     Yəni  h = σ · N + μ.  Bu, μ mərkəzli, σ yayılmalı normal paylanmadan
#     təsadüfi nöqtə seçməyin riyazi yoludur.
#
# NİYƏ BELƏ YAZILIR (birbaşa "buluddan seç" demirik)?
#   Şəbəkənin ÖYRƏNMƏSİ üçün gradient (törəmə) lazımdır. Təsadüfi seçmə
#   əməliyyatının özündən gradient axmır. Hiylə: təsadüfiliyi ayrı N-ə
#   köçürürük (onun öyrənilən parametri yoxdur), μ və σ isə adi
#   toplama/vurma ilə daxil olur. Beləcə gradient μ və σ-ya çatır.
#
# DİQQƏT: Lambda qat öyrətmə və predict zamanı EYNİ işləyir, yəni
# encoder.predict həmişə səs-küylü çıxış verir (aşağıda scatter-ə təsir edir).

h = Lambda(noiser, output_shape=(hidden_dim,))([z_mean, z_log_var])
# Lambda qatı noiser funksiyasını şəbəkəyə qoşur və iki girişi verir.
# output_shape=(2,) → çıxış 2 ədəddir.
# h: hər şəkil üçün gizli fəzada təsadüfi seçilmiş nöqtə (forma (batch, 2)).
# Parametr: yoxdur (öyrənilən çəki yoxdur).
#
# DİQQƏT: h adı burada simvolik tensordur. Aşağıda eyni ad h
# predict nəticəsi (numpy massivi) üçün yenidən istifadə olunur.


# ---------------------------------------------------------------------
# 5) DECODER HİSSƏSİ
# ---------------------------------------------------------------------
#
#   Giriş     Dense+BN+Drop   Dense+BN+Drop    Dense       Reshape
#    (2,)  ─►   (128,)    ─►   (256,)      ─► (784,) ─► (28,28,1)

input_dec = Input(shape=(hidden_dim,))
# Decoder-in ayrıca girişi: 2 ədədlik vektor. Encoder-dən ASILI DEYİL,
# ona görə decoder-ə istənilən 2 ədədi birbaşa verə bilərik (aşağıdakı şəbəkədə).

d = Dense(128, activation='relu')(input_dec)
# 2 → 128, ReLU. Parametr: 2 × 128 + 128 = 384.
# Kiçik koddan yenidən genişlənməyə başlayır. Simmetriya: encoder 256→128,
# decoder 128→256 (güzgü kimi).

d = dropout_and_batch(d)
# BatchNorm: 512 parametr.

d = Dense(256, activation='relu')(d)
# 128 → 256, ReLU. Parametr: 128 × 256 + 256 = 33024.

d = dropout_and_batch(d)
# BatchNorm: 1024 parametr.

d = Dense(28*28, activation='sigmoid')(d)
# 256 → 784, sigmoid (çıxış 0..1, piksel aralığı ilə eyni).
# Parametr: 256 × 784 + 784 = 201488.

decoded = Reshape((28, 28, 1))(d)
# (784,) → (28, 28, 1). Parametr: yoxdur.


# ---------------------------------------------------------------------
# 6) MODELLƏRİN YIĞILMASI
# ---------------------------------------------------------------------

encoder = keras.Model(input_img, h, name='encoder')
# Şəkil girir, h (2 ədəd, səs-küylü) çıxır.
# DİQQƏT: encoder-in çıxışı z_mean DEYİL, h-dır (yəni təsadüfi seçilmiş nöqtə).
# Ona görə eyni şəkli iki dəfə predict etsək, nəticə bir qədər fərqli olur.
# Parametr: 235908 (BatchNorm-ın 768-i öyrənilmir, onlar statistikadır).

decoder = keras.Model(input_dec, decoded, name='decoder')
# 2 ədəd girir, şəkil çıxır. Parametr: 236432.

vae = keras.Model(input_img, decoder(encoder(input_img)), name="vae")
# Bütöv model: şəkil → encoder → h → decoder → şəkil.
# Üç model ÇƏKİLƏRİ PAYLAŞIR. vae-ni öyrətdikdə encoder və decoder də
# öyrənir. Ayrıca fit() çağırmaq lazım deyil.
# Ümumi parametr: 472340 (əvvəlki 2 ölçülü AE-də 160018 idi; BatchNorm və
# daha geniş qatlar fərqi yaradır).


# ---------------------------------------------------------------------
# 7) LOSS FUNKSİYASI (VAE-nin ikinci ürəyi)
# ---------------------------------------------------------------------

def vae_loss(x, y):
    x = tf.reshape(x, shape=(batch_size, 28*28))
    y = tf.reshape(y, shape=(batch_size, 28*28))
    loss = tf.reduce_sum(tf.square(x-y), axis=-1)
    kl_loss = -0.5 * tf.reduce_sum(1 + z_log_var - tf.square(z_mean) - tf.exp(z_log_var), axis=-1)
    return loss + kl_loss
# Loss İKİ hissədən ibarətdir. Burada x = orijinal (y_true), y = bərpa (y_pred).
# (DİQQƏT: bu x, y ad olaraq yuxarıdakı x-dən fərqlidir, yalnız funksiyanın içindədir.)
#
# a) BƏRPA XƏTASI (reconstruction loss):
#      x və y (60, 28, 28, 1) → (60, 784) düzləşdirilir.
#      loss = Σ (x − y)²  → hər şəkil üçün 784 pikselin kvadrat xətalarının CƏMİ.
#    Burada ORTALAMA yox, CƏM istifadə olunur. Əvvəlki kodlarda
#    mean_squared_error (ortalama) idi. Fərq vacibdir: cəm bərpa xətasını
#    ~784 dəfə daha ağır edir, KL ilə müqayisədə.
#    Rolu: "bərpa orijinala nə qədər yaxın olsun".
#
# b) KL-DİVERGENSİYA (kl_loss):
#      -0.5 · Σ ( 1 + log σ² − μ² − σ² )
#    Rolu: hər şəklin buludunu (μ, σ) standart normal N(0, 1)-dən çox
#    UZAQLAŞMAMAĞA məcbur edir.
#      - μ² termi → mərkəzləri 0-a çəkir.
#      - σ² − log σ² − 1 termi → yayılmanı 1-ə yaxın saxlayır
#        (σ=1 olanda bu termin ən kiçik qiyməti 0-dır).
#    Rolu: gizli fəzanı nizamlayır, boşluqları aradan qaldırır.
#    z_mean və z_log_var burada GLOBAL dəyişənlərdir (noiser-də mənimsədilib).
#
# İKİSİ ARASINDA DAVA:
#   Bərpa xətası istəyir: hər şəkil öz ayrı, kəskin, dar nöqtəsində dayansın
#                         (dəqiq bərpa üçün).
#   KL istəyir:           hamısı 0 ətrafında, geniş, bir-birinə qarışıq olsun.
#   Kompromis: oxşar rəqəmlər yan-yana, hamısı birlikdə [-3, 3] ətrafında.
#
# Funksiya hər şəkil üçün forma (60,) vektor qaytarır. Keras sonra ortalama alır.

vae.compile(optimizer='adam', loss=vae_loss)
# Adam optimizer, defolt öyrənmə sürəti 0.001. Loss — yuxarıdakı özümüz yazdığımız.
# metrics yoxdur.


# ---------------------------------------------------------------------
# 8) ÖYRƏTMƏ
# ---------------------------------------------------------------------

vae.fit(x_train, x_train, epochs=5, batch_size=batch_size, shuffle=True)
# fit(x_train, x_train): giriş də, cavab da eyni şəkildir (avtoenkoder məntiqi).
#
# epochs=5       → bütün 60000 şəkil 5 dəfə keçilir.
# batch_size=60  → hər addımda 60 şəkil; epoch-da 1000 addım, cəmi 5000 yeniləmə.
# shuffle=True   → hər epoch-da sıra qarışdırılır.
#
# NƏ GÖZLƏMƏK LAZIMDIR? Loss iki hissənin cəmi olduğu üçün və cəm istifadə
# etdiyimiz üçün rəqəmlər əvvəlki koddan (0.0x) qat-qat BÖYÜK görünəcək
# (onlarla). Bu pis demək deyil: miqyas fərqlidir. Epoch-lar keçdikcə
# azalmalıdır. Dəqiq rəqəm hər işlətmədə dəyişir.
# Dropout(0.3) öyrətmədə loss-u bir az yüksək saxlayır.


# ---------------------------------------------------------------------
# 9) GİZLİ FƏZANIN GÖRÜNTÜSÜ
# ---------------------------------------------------------------------

h = encoder.predict(x_test[:6000], batch_size=batch_size)
# İlk 6000 test şəklini encoder-dən keçirir. Forma: (6000, 2).
#
# NİYƏ 6000, 10000 yox? Sürət üçün; həm də 6000 % 60 == 0 (yarımçıq batch
# olmasın, çünki noiser sabit forma tələb edir).
# NİYƏ batch_size=60 verilir? predict-in defolt batch-i 32-dir. Bu isə
# noiser-in (60, 2) sabit forması ilə uyğun gəlməzdi, xəta çıxardı.
#
# DİQQƏT: h burada artıq numpy massivdir (yuxarıdakı simvolik h yox).
# Bu, Lambda qatın SƏS-KÜYLÜ çıxışıdır (z_mean yox). Ona görə nöqtələr
# bir qədər "yayılmış" görünür.

plt.scatter(h[:, 0], h[:, 1])
# 6000 nöqtəni müstəvidə çəkir (x oxu = 1-ci koordinat, y oxu = 2-ci).
# Nə görmək lazımdır?
#   - Nöqtələr təxminən 0 ətrafında bir dairəvi/oval bulud kimi yığılır;
#     əksər nöqtələr təxminən -3..3 aralığında olur. Bu, KL-in təsiridir.
#     Əvvəlki AE-də isə aralıq nəzarətsiz idi (0..300 ola bilərdi).
#   - Boşluqlar azdır, nöqtələr bütöv bir bulud əmələ gətirir.
# Rəng yoxdur. Hansı rəqəmin harada olduğunu görmək üçün y_test ilə rəngləmək olar:
#   plt.scatter(h[:, 0], h[:, 1], c=y_test[:6000], cmap='tab10', s=2)
#   plt.colorbar()
# DİQQƏT: plt.show() yoxdur. Notebook-da qrafik avtomatik görünür.
# Adi .py faylında isə pəncərə açılmaz, plt.show() lazımdır.


# ---------------------------------------------------------------------
# 10) GENERASİYA: GİZLİ FƏZANIN ŞƏBƏKƏSİ ÜZRƏ GƏZİNTİ
# ---------------------------------------------------------------------

n = 5
# Hər istiqamətdə 5 addım. Mərkəzdən hər tərəfə 5, yəni -5..5.

total = 2*n+1
# 2·5 + 1 = 11. Şəbəkə 11 × 11 = 121 şəkildən ibarət olacaq.

plt.figure(figsize=(total/2, total/2))
# YENİ qrafik pəncərəsi açır (11×11 düym).
# Bu sətir scatter-dən sonra olduğu üçün scatter və şəkillər ÜST-ÜSTƏ
# düşmür (əvvəlki koddakı tuzaq burada yoxdur): scatter ayrı, şəbəkə ayrı.

num = 1
# plt.subplot-un nömrəsi 1-dən başlayır (soldan sağa, yuxarıdan aşağı).

for i in range(-n, n+1):
    for j in range(-n, n+1):
        # i və j: -5, -4, ..., 0, ..., 5. Hər cüt gizli fəzada bir nöqtəni təyin edir.
        # i xarici dövr → SƏTİR, j daxili dövr → SÜTUN.
        # Yəni sətirlər 1-ci latent koordinata (i), sütunlar 2-ci koordinata (j) uyğundur.

        ax = plt.subplot(total, total, num)
        # 11×11 şəbəkədə num-cu xananı seçir.

        num += 1

        img = decoder.predict(np.expand_dims([3*i/n, 3*j/n], axis=0))
        # Nöqtə: [3·i/5, 3·j/5]. i=-5 → -3, i=0 → 0, i=5 → +3.
        # Yəni koordinatlar -3-dən +3-ə qədər 0.6 addımla gedir.
        # NİYƏ 3? VAE sayəsində nöqtə buludu təxminən N(0,1)-dir; 3σ
        # aralığı onun demək olar ki, hamısını əhatə edir.
        # (Əvvəlki koddakı [50, 250] kimi "təsadüfi" rəqəmlər artıq lazım deyil.)
        #
        # np.expand_dims(..., axis=0): (2,) → (1, 2), Keras batch gözləyir.
        # decoder.predict: çıxış (1, 28, 28, 1).
        #
        # DİQQƏT: 121 dəfə ayrı-ayrı predict çağırılır; bu YAVAŞDIR.
        # Sürətli yol: bütün 121 nöqtəni bir massivə yığıb BİR dəfə predict etmək.
        #
        # decoder-də noiser YOXDUR, ona görə burada təsadüfilik yoxdur:
        # eyni nöqtə həmişə eyni şəkli verir. BatchNorm/Dropout predict-də
        # öyrətmə rejimindən fərqli (sabit) işləyir.

        plt.imshow(img.squeeze(), cmap='gray')
        # .squeeze(): (1, 28, 28, 1) → (28, 28). cmap='gray': boz ton.

        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        # Oxları gizlədir ki, şəbəkə təmiz görünsün.

# NƏTİCƏ: 11×11 şəkil cədvəli. Gözləmək olar:
#   - Qonşu xanalarda rəqəmlər TƏDRİCƏN dəyişir (məsələn, 1 → 7 → 9 → 4...).
#     Bu, gizli fəzanın "hamar" olduğunu göstərir (VAE-nin əsas üstünlüyü).
#   - Şəkillər bulanıq olacaq (2 ölçü + 5 epoch + KL nizamlaması).
#   - Mərkəzə yaxın xanalar adətən daha aydın, kənarlar daha pis olur.
#
# SON QEYDLƏR:
#   1) Sonda plt.show() əlavə edin (notebook-da lazım deyil, .py-də lazımdır).
#   2) Hər işlətmədə çəkilər təsadüfi başladığı üçün hansı rəqəmin şəbəkənin
#      harasında olması DƏYİŞİR, amma ümumi struktur (hamar keçidlər,
#      [-3, 3] aralığı) qalır.
#   3) Daha yaxşı keyfiyyət üçün: epoch-ları artırmaq, hidden_dim-i 2-dən
#      böyük etmək və ya bərpa xətasını ortalama kimi yazıb KL-ə çəki vermək olar.